# PLOT STACKED SPECTRA

In [ ]:
from IPython.display import display, HTML

def expand_output():
    display(HTML("""
    <style>
    .jp-OutputArea-output {
        height: auto !important;
        max-height: none !important;
        overflow: visible !important;
    }
    </style>
    """))

expand_output()

In [ ]:
from spectraPyle.plot.plot import plotting, plot_h5_heatmap

In [ ]:
# Example filename (new format since v5.1):
# <SURVEY>_<GRISMS>_<DR>_<CATALOG>_<ZCOL>_<NORM>[_<L0>-<L1>]_s<SIGMA>_<PXTOKEN>_<HASH8>.fits
name_stack = "/Users/salvatorequai/Documents/EUCLID/DR1/spettri_test/output/deep_red-blue_DR1_test_vis_insp_no_duplicates_spe_z_med_s3_px3lin_74d046f6.fits"

In [ ]:
plotting(name_stack, width=850, height=550)

**Having trouble seeing the plot? A quick browser refresh should do the trick!**

---
## 2D HEATMAP / THIN LINES — H5 array viewer

In [ ]:
import ipywidgets as widgets
from IPython.display import display
from pathlib import Path

# ── Paths ──────────────────────────────────────────────────────────────────
_fits_p   = Path(name_stack).expanduser()
fits_path = str(_fits_p)
h5_path   = str(_fits_p.parent / (_fits_p.stem + '_array.h5'))

# ── Available metrics from the FITS ────────────────────────────────────────
from astropy.io import fits as astrofits
with astrofits.open(fits_path) as hdul:
    all_cols = list(hdul['STACKING_RESULTS'].columns.names)
    wavelength_col = hdul['STACKING_RESULTS'].data['wavelength']
    n_pixels = len(wavelength_col)

metric_choices = [c for c in all_cols
                  if c.startswith('spec')
                  and not any(s in c for s in ('Error','Dispersion','16th','84th','98th','99th'))]

# ── Widgets ────────────────────────────────────────────────────────────────
w_mode     = widgets.ToggleButtons(options=['heatmap','lines'],  description='Mode',    value='heatmap')
w_template = widgets.ToggleButtons(options=['original','norm'],  description='Array',   value='original')
w_metric   = widgets.Dropdown(     options=metric_choices,       description='Metric',  value='specMedian')
w_normfact = widgets.Checkbox(     value=False,                  description='Norm factors')
w_maxspec  = widgets.BoundedIntText(value=0, min=0, max=100000, step=100,
                                    description='Max spectra (0=all, random)')
w_alpha    = widgets.FloatSlider(  value=0.05, min=0.01, max=0.5, step=0.01,
                                   description='Line alpha',
                                   layout=widgets.Layout(width='400px'))
w_binsx    = widgets.IntSlider(    value=n_pixels, min=10, max=n_pixels, step=10,
                                   description=f'X bins (max={n_pixels})',
                                   layout=widgets.Layout(width='400px'))
w_binsy    = widgets.IntSlider(    value=200, min=10, max=500, step=10,
                                   description='Y bins',
                                   layout=widgets.Layout(width='400px'))
w_button   = widgets.Button(description='Plot', button_style='primary',
                             layout=widgets.Layout(width='120px'))
#out = widgets.Output()
out = widgets.Output(layout=widgets.Layout(width='100%', height='100%'))

def on_plot(_):
    out.clear_output(wait=True)
    with out:
        max_s = w_maxspec.value if w_maxspec.value > 0 else None
        fig = plot_h5_heatmap(
            h5_path        = h5_path,
            fits_path      = fits_path,
            template_array = w_template.value,
            metric         = w_metric.value,
            norm_factors   = w_normfact.value,
            mode           = w_mode.value,
            max_spectra    = max_s,
            line_alpha     = w_alpha.value,
            nbinsx         = w_binsx.value,
            nbinsy         = w_binsy.value,
        )
        display(fig)

w_button.on_click(on_plot)

In [ ]:
display(widgets.VBox([
    widgets.HBox([w_mode, w_template]),
    widgets.HBox([w_metric, w_normfact]),
    widgets.HBox([w_maxspec, w_alpha]),
    widgets.HBox([w_binsx, w_binsy]),
    w_button,
    
]))

display(out,)